[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI/blob/main/notebooks/03_using_an_llm.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [15]:
# === Colab-compat setup (no-op when running locally) ===
# This whole block only DOES something on Google Colab. Locally, IN_COLAB is
# False, so the if-block is skipped entirely and only the threading caps run.
import os, sys

# Detect Colab by checking whether its special module has been imported.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab the repo isn't present, so we clone it, cd into the notebooks
    # folder, and install the libraries Colab doesn't ship with.
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
# setdefault means: only set the env var if it isn't already defined.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless warning
# (Old OMP/MKL one-thread caps removed — they slowed CPU-only machines.)

# L10 · NB 03 — Using a small LLM

> ⏱️ **~40 min** core (Sections 1–9) · +15 min for the 🟡 Extension · slow on CPU? use the Colab T4 badge above

> *We have the architecture. Now let's load a generative transformer and have it produce text. Same Lego bricks as NB 02 — but trained to predict the next token, with a causal mask and an instruction-tuning phase.*

Today's model: `SmolLM2-360M-Instruct` from Hugging Face. 361M parameters, ~720 MB on disk, runs on CPU in 1-3 seconds per response.

In this notebook we will:

1. Load the model and its tokenizer
2. Watch tokenisation happen end-to-end
3. Generate text — first with greedy decoding, then with sampling
4. Tweak temperature, top-k, top-p and see how they change output
5. Use chat templates the right way for instruction-tuned models

---

### Where you are in L10

Marcus asked for a shopping assistant. Sarah can't build it in one leap — she works through four notebooks, each answering the next question in the chain:

| Step | Notebook | Sarah's question |
|---|---|---|
| 1 | `01_monday_morning` | What can pretrained transformers already do, out of the box? |
| 2 | `02_attention_intuition` | *Why* do they work? What is this "attention" everyone talks about? |
| 3 | `03_using_an_llm` 👈 **you are here** | How do I drive a generative LLM myself — tokens, sampling, chat? |
| 4 | `04_rag_pipeline` | How do I ground it in NorthStar's catalogue? → the shopping assistant |

Right now you're on **step 3 of 4**. Each notebook stands on the one before it — run them in order.


## 1 · Setup


> **Why this cell first — and why it looks like a pile of switches.**
> Before Sarah touches a single model she runs this setup cell. It isn't glamorous, but every line earns its place:
>
> - `KMP_DUPLICATE_LIB_OK='TRUE'` — on macOS, PyTorch and the other scientific libraries each ship their own copy of the OpenMP maths runtime. When two copies load at once the kernel **crashes the instant you `import torch`**. This line tells them to coexist instead of fighting. (It's the same fix the repo's `.env` file applies — we repeat it here so the notebook also works in Colab and on a fresh machine.)
> - `HF_HUB_DISABLE_TELEMETRY` / `TRANSFORMERS_VERBOSITY='error'` / `warnings.filterwarnings('ignore')` — silence the download chatter and version warnings so the output you see is the output that *matters*.
>
> You'll see this same opening cell in every model notebook. Run it once at the top — it's the seatbelt before the drive.


In [16]:
# --- Environment guards (set BEFORE importing torch/transformers) ---
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home to Hugging Face
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # only show errors, hide info/warning chatter

import warnings
warnings.filterwarnings('ignore')  # mute version/deprecation warnings for cleaner output

# Core imports: time (timing), torch (the tensor engine), and the two Hugging Face
# "Auto" classes that pick the right tokenizer/model for any checkpoint name.
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)
torch.manual_seed(0)      # fix the random seed so runs are reproducible

# The model checkpoint we'll use: a small (361M-param) instruction-tuned LLM.
MODEL_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'

# Is the model already on disk? from_pretrained() downloads on the first run and
# reuses the local cache afterwards, but it does so silently. This check just tells
# us which is about to happen: a ~720 MB download, or an instant load from cache.
# (Cache lives in ~/.cache/huggingface by default; override with the HF_HOME env var.)
from huggingface_hub import try_to_load_from_cache

# Returns the local file path (a str) if config.json is cached, else a sentinel/None.
cached = try_to_load_from_cache(MODEL_NAME, 'config.json')
if isinstance(cached, str):
    print(f"'{MODEL_NAME}' already downloaded - loading from local cache.")
else:
    print(f"'{MODEL_NAME}' not found in cache - downloading (~720 MB, first run only)...")

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
# The tokenizer turns text <-> token IDs. It MUST match the model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# AutoModelForCausalLM = a "predict the next token" (causal/decoder) language model.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"Loaded in {time.time() - t0:.1f}s")
# numel() counts elements in each weight tensor; summing gives total parameters.
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Vocab size = how many distinct tokens the model knows (size of its output layer).
print(f"Vocab size: {tokenizer.vocab_size:,}")

'HuggingFaceTB/SmolLM2-360M-Instruct' already downloaded - loading from local cache.
Loading HuggingFaceTB/SmolLM2-360M-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded in 2.2s
Parameters: 361,821,120
Vocab size: 49,152


## 2 · Tokenisation, step by step

Before the model sees any text, the tokenizer chops it into subword IDs. Let's see exactly what happens for one prompt.

In [17]:
text = "What's a good summer dress under £80?"

# Step 1: text -> token strings (for human readability)
# .tokenize() shows the subword PIECES the text is split into (Ġ marks a leading space).
token_strs = tokenizer.tokenize(text)
print(f"Input text: {text!r}")
print(f"Tokens    : {token_strs}")

# Step 2: tokens -> integer IDs (what the model actually consumes)
# .encode() maps each subword to its integer ID in the vocabulary. The model
# never sees letters — only these numbers.
token_ids = tokenizer.encode(text)
print(f"IDs       : {token_ids}")
print(f"Length    : {len(token_ids)} tokens")

Input text: "What's a good summer dress under £80?"
Tokens    : ['What', "'s", 'Ġa', 'Ġgood', 'Ġsummer', 'Ġdress', 'Ġunder', 'ĠÂ£', '8', '0', '?']
IDs       : [1780, 506, 253, 1123, 4018, 10272, 656, 10257, 40, 32, 47]
Length    : 11 tokens


Observations:
- Common words → one token each (`What`, `summer`)
- Possessives split (`'s`)
- Punctuation gets its own token
- Currency symbols like `£` may split into a few subword pieces depending on vocab

Every model has its OWN tokenizer. The same input string produces different token sequences for SmolLM2 vs MiniLM vs GPT-2 vs Llama. Don't mix tokenizers across models.

## 3 · The simplest generation — one forward pass

Before we wrap things up in `.generate()`, let's see what ONE forward pass looks like. Feed token IDs, get **logits** over the vocabulary for the NEXT token.

> **What's a logit?**
> A logit is the **raw, unnormalised score** the model assigns to each possible next token — one number per word in its 49K-token vocabulary. Higher = the model likes that token more as the continuation. But logits are *not* probabilities: they can be negative, they don't fall between 0 and 1, and they don't add up to 1. They're just the model's un-tidied opinion.
>
> To turn them into real probabilities we apply **softmax**, which does two things: it makes every value positive, and it rescales them so they sum to 1.0. So the pipeline is:
>
> ```
> token IDs  ──model──▶  logits  ──softmax──▶  probabilities  ──pick one──▶  next token
>            (49K raw scores)     (49K values that sum to 1)
> ```
>
> In the output below, watch the two columns: `logit` is the raw score (e.g. `18.4`, `15.1`, …) and `prob` is that same score after softmax (e.g. `0.62`, `0.11`, …). Same ranking, but `prob` is now human-readable — "the model is 62% sure the next token is X."

In [18]:
# Calling the tokenizer (vs .encode) returns a dict of tensors ready for the model.
# return_tensors='pt' -> PyTorch tensors; it also adds a batch dimension.
inputs = tokenizer(text, return_tensors='pt')
# input_ids shape is (batch, seq_len): one row per prompt, one column per token.
print(f"Input IDs shape: {inputs['input_ids'].shape}  (batch=1, seq_len)")

# One forward pass. no_grad() skips gradient tracking — we're predicting, not training.
with torch.no_grad():
    # **inputs unpacks input_ids AND attention_mask (1 = real token, 0 = padding).
    outputs = model(**inputs)

# logits = raw, unnormalised scores — one per vocab token, at EVERY position.
logits = outputs.logits   # shape: (batch, seq_len, vocab_size)
print(f"Logits shape   : {logits.shape}")

# For "what comes next", we only care about the prediction at the LAST position.
# [0, -1, :] = batch 0, last token, all vocab scores.
next_token_logits = logits[0, -1, :]
print(f"Next-token logits shape: {next_token_logits.shape}  (one number per vocab token)")

# topk finds the 5 highest-scoring candidate next tokens.
top5 = torch.topk(next_token_logits, 5)
print(f"\nTop 5 candidates for the next token:")
for prob_logit, tok_id in zip(top5.values, top5.indices):
    # softmax turns the raw logits into probabilities (0-1, summing to 1).
    prob = torch.softmax(next_token_logits, dim=-1)[tok_id]
    print(f"  {tokenizer.decode(tok_id):<20s}  logit={prob_logit:>6.2f}  prob={prob:.3f}")

Input IDs shape: torch.Size([1, 11])  (batch=1, seq_len)
Logits shape   : torch.Size([1, 11, 49152])
Next-token logits shape: torch.Size([49152])  (one number per vocab token)

Top 5 candidates for the next token:
  <|im_end|>            logit= 18.38  prob=0.711
  
                     logit= 16.38  prob=0.096
   I                    logit= 15.88  prob=0.059
   
                    logit= 14.44  prob=0.014
   �                    logit= 14.19  prob=0.011


That's the entire model: 360M parameters processed your input and produced a probability distribution over 49K vocabulary tokens. Pick one, append it, run again. That's text generation.

## 4 · The generation loop, by hand

`model.generate()` does this loop for us. Let's first do it manually to see exactly what's happening.

In [19]:
prompt = "The best way to learn programming is"
# Grab just the input_ids tensor (we'll keep appending to it as we generate).
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
print(f"Prompt: {prompt!r}")
print(f"Starting with {input_ids.shape[1]} tokens.")

# Greedy generation: pick the argmax token each step
max_new = 30
for step in range(max_new):
    with torch.no_grad():
        out = model(input_ids)  # forward pass on the whole sequence so far
    # argmax = take the single highest-probability next token (deterministic, "greedy").
    next_id = out.logits[0, -1, :].argmax().item()
    # append the chosen token to the running sequence, then loop and predict again
    input_ids = torch.cat([input_ids, torch.tensor([[next_id]])], dim=1)
    # stop on EOS — the model's "I'm done" end-of-sequence token
    if next_id == tokenizer.eos_token_id:
        break

# Decode IDs back to text; skip_special_tokens drops markers like <|im_end|>.
generated = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(f"\nGenerated:\n  {generated}")

Prompt: 'The best way to learn programming is'
Starting with 7 tokens.

Generated:
  The best way to learn programming is to start with a language that is easy to learn and has a large community of users. Python is a great language to start with because it is:


That's the entire generation algorithm. Every modern LLM — GPT-4, Claude, Llama 70B — does fundamentally this. The model is bigger and trained differently, but the loop is identical.

## 5 · Use `model.generate()` for convenience

Doing the loop manually is illustrative but slow. The library's `.generate()` handles batching, caching, and stopping conditions efficiently.

In [20]:
prompt = "The best way to learn programming is"
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']

t0 = time.time()
# .generate() runs the same predict-append loop internally, but fast (with KV caching).
output_ids = model.generate(
    input_ids,
    max_new_tokens=40,      # cap on how many NEW tokens to produce
    do_sample=False,        # greedy — always take the argmax (no randomness)
    pad_token_id=tokenizer.eos_token_id,  # use EOS as padding to silence a warning
)
elapsed = time.time() - t0

generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Generated in {elapsed:.2f}s:\n  {generated}")

Generated in 1.49s:
  The best way to learn programming is to start with a language that is easy to learn and has a large community of users. Python is a great language to start with because it is:

Easy to learn: Python has a simple


## 6 · Sampling — temperature, top-k, top-p

Greedy is deterministic and produces boring text. Real applications use sampling. Let's vary the parameters and watch the output change for the same prompt.

In [21]:
# Instruction-tuned models expect chat-formatted prompts. Raw text often
# makes them emit EOS immediately and produce empty output — see Section 7
# of this notebook for the wrong-vs-right comparison. Use apply_chat_template.
messages = [{'role': 'user', 'content': 'Tell me a short bedtime story about a brave little robot.'}]
# apply_chat_template wraps the message in the special turn tokens the model was trained on.
chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(chat_prompt, return_tensors='pt')['input_ids']

# Three decoding settings, from deterministic (greedy) to high-randomness.
configs = [
    ('Greedy',              dict(do_sample=False)),                             # no randomness
    ('Temp=0.7, top-p=0.9', dict(do_sample=True, temperature=0.7, top_p=0.9)),  # balanced
    ('Temp=1.5, top-p=0.95',dict(do_sample=True, temperature=1.5, top_p=0.95)), # wild/creative
]

for name, cfg in configs:
    torch.manual_seed(42)  # consistent randomness across configs
    out = model.generate(
        # temperature: <1 sharpens the distribution (safer), >1 flattens it (riskier/creative).
        # top_p (nucleus): only sample from the smallest set of tokens whose probs sum to p.
        # do_sample=True turns on random sampling; False = greedy argmax.
        input_ids, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id, **cfg
    )
    # Slice off the prompt tokens so we print only the newly generated continuation.
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n=== {name} ===")
    print(text.strip()[:300])


=== Greedy ===
Once upon a time, in a faraway land, there was a tiny robot named Robby. Robby was a brave little robot who loved to explore the world. He had a special suit that made him invisible, so he could sneak around and do all sorts of exciting things.

Robby

=== Temp=0.7, top-p=0.9 ===
Once upon a time, in a place called Robotville, there lived a tiny robot named Robby. Robby was a brave little robot who loved to play with his friends. One day, he decided to go on a mission to save the city from a very big and scary robot named Robo

=== Temp=1.5, top-p=0.95 ===
Livabo lived far away from here - it is up near North Carolina! Livabo didn't need to worry too much anymore though, for in these days lived a big and busy robot that was more than ten times smaller. One sunny afternoon and when Livabo came to his home and found himself


**Notice how temperature controls creativity vs coherence.** Greedy is safest but repetitive. T=1.5 starts to drift but is more interesting. Production sweet spot: T=0.7-0.8 + top-p=0.9 for general use; T=0.2 for code; T=1.0 for creative writing.

Be aware: a 360M model has limits. Don't read too much into the *content* — focus on how each parameter shifts the distribution.

## 7 · Chat templates — what instruction-tuned models actually expect

`SmolLM2-360M-Instruct` was fine-tuned on conversations, so it expects your text wrapped in a **specific format** with special tokens marking each turn (`<|im_start|>user ... <|im_end|>`).

The easiest way to understand why this matters: ask the SAME question both ways and compare the answers.

In [22]:
question = "What is a CNN?"

# --- WITHOUT the chat template: send the raw question as-is ---
raw_ids = tokenizer(question, return_tensors='pt')['input_ids']
raw_out = model.generate(raw_ids, max_new_tokens=80, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
without = tokenizer.decode(raw_out[0][raw_ids.shape[1]:], skip_special_tokens=True)

# --- WITH the chat template: wrap the question in the format the model was trained on ---
# apply_chat_template adds the <|im_start|>/<|im_end|> turn markers and the
# "assistant" prompt so the model knows it's now its turn to answer.
messages = [{"role": "user", "content": question}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
chat_ids = tokenizer(formatted, return_tensors='pt')['input_ids']
chat_out = model.generate(chat_ids, max_new_tokens=80, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
with_template = tokenizer.decode(chat_out[0][chat_ids.shape[1]:], skip_special_tokens=True)

# --- Compare the two answers side by side ---
print("QUESTION:", question)
print("\n=== WITHOUT chat template ===")
print(without.strip() or "(model returned almost nothing)")
print("\n=== WITH chat template ===")
print(with_template.strip())

QUESTION: What is a CNN?

=== WITHOUT chat template ===
(model returned almost nothing)

=== WITH chat template ===
A Convolutional Neural Network (CNN) is a type of artificial neural network that is particularly well-suited for image and video processing tasks. It's a type of neural network that uses convolutional layers, pooling layers, and fully connected layers to extract features from input data, such as images or videos.

Convolutional layers are designed to process data at the level of image or video features


**Same question, very different results.** Without the template the model doesn't recognise a "user turn" and drifts or stops early; with the template it gives a clean, focused answer.

**Always use `apply_chat_template`** for instruction-tuned models — don't hand-roll the special tokens. The model's training defined exactly which tokens delimit each turn.

## 8 · Multi-turn conversation

Here's something that surprises people: **the model has no memory.** Each call to `generate()` starts from a blank slate — it doesn't recall your previous questions. So how do chatbots hold a conversation?

**The trick: you resend the entire history every turn.** "Memory" is an illusion created by the application, not a feature of the model. Before each new question, we paste the whole transcript back in — every user message and every assistant reply so far — and append the new question at the end. The model re-reads the full conversation from scratch and continues it.

```
Turn 1 sends:  [system] [user: Q1]
Turn 2 sends:  [system] [user: Q1] [assistant: A1] [user: Q2]   ← Q1 and A1 resent
Turn 3 sends:  [system] [user: Q1] [assistant: A1] [user: Q2] [assistant: A2] [user: Q3]
```

That growing `conversation` list below IS the memory. The model looks "smart" only because we keep handing it the full script.

> **What is the context window?**
> Every model can only read a fixed maximum number of **tokens** in one go — that limit is its **context window**. It covers *everything* the model sees at once: the system prompt + the entire conversation history + the new question + the room it needs to write its answer.
>
> - SmolLM2 here: ~8,192 tokens. GPT-4-class models: 128,000+. Some newer models: 1,000,000+.
> - Because we resend the whole history each turn, the transcript **keeps growing** — and once it exceeds the context window, the model physically cannot see the earliest messages. That's why long chatbot sessions "forget" the beginning of the conversation.
> - Real apps deal with this by **truncating** old turns or **summarising** them to stay under the limit.
>
> Keep this in mind for NB 04: RAG works by stuffing relevant documents into this same limited window — so context-window size directly limits how much reference material you can give the model at once.

In [23]:
# A multi-turn conversation: the list of messages IS the model's memory.
conversation = [
    {"role": "system", "content": "You are a helpful retail shopping assistant. Answer concisely."},  # sets behaviour/persona
    {"role": "user", "content": "What's a good fabric for a summer dress?"},
]

def chat(messages, max_new_tokens=120):
    # Format the full history, generate, and return only the new assistant text.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

response = chat(conversation)
print("Assistant:", response)

# Add it to the history and continue
# We append the model's reply AND the next user question, so turn 2 sees turn 1.
conversation.append({"role": "assistant", "content": response})
conversation.append({"role": "user", "content": "Is linen better than cotton for hot weather?"})

response = chat(conversation)  # now the model has the full context to answer the follow-up
print("\nAssistant:", response)

Assistant: A good fabric for a summer dress is cotton or linen.

Assistant: Yes, linen is better for hot weather.


**Notice three things:**

1. The **system prompt** sets behaviour ("be a retail assistant, be concise"). You can shape style and constraints here.
2. We **append** the assistant's reply to the conversation history before the next user turn. That's how the model "remembers" what it said.
3. The model gives reasonable, fluent retail-style answers — for a 360M-parameter model, that's remarkable. Imagine what 70B can do.

This is the loop that powers every chatbot, from a customer-service bot to ChatGPT. The model is bigger; the loop is the same.

## 9 · Recap

You now know:

- Tokenisation: text → subword IDs (model-specific)
- Forward pass: IDs → logits → next-token probability
- Generation: iteratively sample/argmax + append until done
- Sampling: temperature, top-k, top-p — tune the creativity dial
- Chat templates: format prompts the way the model expects
- Multi-turn: append history; LLMs are stateless, you supply the state

**Next:** an LLM by itself doesn't know YOUR data. To make it useful for product-specific Q&A, we need to give it the right context. That's RAG. NB 04.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · Streaming output

For interactive applications you want tokens to appear AS the model generates them — not in one big chunk at the end. `TextStreamer` handles this.

In [24]:
# TextStreamer prints tokens to the console AS they're generated, instead of
# waiting for the whole response — that's how chat UIs show text "typing out".
from transformers import TextStreamer

prompt = "Write a haiku about cosy autumn knitwear:"
messages = [{"role": "user", "content": prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(formatted, return_tensors='pt')['input_ids']

# skip_prompt=True: don't re-print the input; skip_special_tokens=True: hide markers.
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
# Pass streamer= to generate; the return value is ignored (_) since it's printed live.
_ = model.generate(input_ids, max_new_tokens=60, do_sample=True, temperature=0.8,
                   top_p=0.9, pad_token_id=tokenizer.eos_token_id, streamer=streamer)
print('\n[end of generation]')

Soft woolen sweater, snugly warm,
In the cozy winter season, snug and cool,
A perfect winter's gift, the fleece.

[end of generation]


## E2 · Stop sequences

You can tell `.generate()` to stop at a specific token sequence. Useful when you want to constrain output format (e.g., 'stop at a closing bracket').

In [25]:
# Stop generation when the model emits the string "STOP_HERE"
prompt_text = "List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.\n1."
input_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids']

# Find the token IDs for the stop string (might be multiple tokens)
# A stop "sequence" in text can map to several token IDs, so we match the whole list.
stop_ids = tokenizer.encode("STOP_HERE", add_special_tokens=False)
print(f"Stop token IDs: {stop_ids}")

# A simple per-step custom stop function via manual loop
out = input_ids.clone()
for _ in range(80):
    with torch.no_grad():
        nxt = model(out).logits[0, -1, :].argmax().item()  # greedy next token
    out = torch.cat([out, torch.tensor([[nxt]])], dim=1)    # append it
    # Check if the last few tokens match the stop sequence
    # Compare the tail of the output against stop_ids; if equal, we're done.
    if out[0][-len(stop_ids):].tolist() == stop_ids:
        break

print('Generated:')
print(tokenizer.decode(out[0], skip_special_tokens=True))

Stop token IDs: [48124, 79, 56, 14298]
Generated:
List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.
1. *Feta cheese*
2. *Tomato salad*
3. *Balsamic vinaigrette*
4. *Cucumber*
5. *Avocado*
6. *Parmesan cheese*
7. *Bread*
8. *Balsamic glaze*
9. *Ginger*
10. *Cel


Production frameworks (vLLM, llama.cpp) make stop sequences first-class. The mechanic is the same: watch the running output, break when you see what you wanted.